In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

NOMBRE_GRUPO = "energia-y-sustentabilidad-1"

try:
    spark.stop()
except Exception:
    pass

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [ ]:
df.printSchema()
df.show(5, truncate=False)

In [ ]:
df_filtrado = df.filter(
    col("item").isNotNull() &
    col("categoria").isNotNull() &
    (col("valor") > 0)
)

print("Limpieza completada")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

In [ ]:
df_filtrado.select("item", "valor").show(10, truncate=False)
df_filtrado.filter(col("valor") > 40).show(10, truncate=False)
df_filtrado.groupBy("grupo").count().show()

In [ ]:
reporte_categorias = df_filtrado.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.round(F.avg("valor"), 2).alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA")
reporte_categorias.show(truncate=False)

ganador = df_filtrado.orderBy(F.desc("valor")).limit(1)
print("PRODUCTO MAS CARO DE LA BASE")
ganador.select("item", "valor", "categoria", "grupo", "fecha_captura").show(truncate=False)

In [ ]:
ticket_salida = df_filtrado.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show(truncate=False)